In [0]:
# CELL 0 - look at the file before writing the reader
LANDING = "/Volumes/lumora/raw/landing"

raw = dbutils.fs.head(f"{LANDING}/ads/tiktok/tiktok_ads_2024_06.csv", 500)
print(raw)
print("---")
print(repr(raw[:80]))   # repr() makes invisible characters visible (look for \ufeff)

In [0]:
# CELL 1 - Auto Loader: TikTok ads -> bronze
from pyspark.sql import functions as F
import re, unicodedata

LANDING     = "/Volumes/lumora/raw/landing"
CHECKPOINTS = "/Volumes/lumora/raw/checkpoints"
SOURCE      = "tiktok_ads"

def clean_names(df):
    """Make column names valid for Delta: strip BOM, fold accents to ASCII,
    non-alphanumeric -> '_', lowercase. Columns starting with '_' are ours
    (metadata) and are left alone. Values are untouched."""
    for old in df.columns:
        if old.startswith("_"):        # _rescued_data and friends keep their prefix
            continue
        s = old.replace("\ufeff", "")
        s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
        new = re.sub(r"_+", "_", re.sub(r"[^0-9a-zA-Z]+", "_", s)).strip("_").lower()
        if new != old:
            df = df.withColumnRenamed(old, new)
    return df

# readStream = a DataFrame representing data that arrives over time
df = (
    spark.readStream
        .format("cloudFiles")                                  # Auto Loader
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", f"{CHECKPOINTS}/{SOURCE}/schema")
        .option("cloudFiles.inferColumnTypes", "false")        # everything as STRING
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("header", "true")
        .option("sep", ";")                                    # pt-BR export
        .option("encoding", "UTF-8")                           # BOM on the first header
        .option("rescuedDataColumn", "_rescued_data")          # nothing lost silently
        .load(f"{LANDING}/ads/tiktok/")
)

df = clean_names(df)

# Lineage: WHERE each row came from and WHEN we read it
df = (
    df.withColumn("_source_file",   F.col("_metadata.file_path"))
      .withColumn("_file_modified", F.col("_metadata.file_modification_time"))
      .withColumn("_ingested_at",   F.current_timestamp())
      .withColumn("_source_system", F.lit("tiktok_ads"))
)

# trigger(availableNow=True): process everything pending, then stop
query = (
    df.writeStream
      .option("checkpointLocation", f"{CHECKPOINTS}/{SOURCE}/checkpoint")
      .option("mergeSchema", "true")
      .trigger(availableNow=True)
      .toTable("lumora.bronze.tiktok_ads_raw")
)

query.awaitTermination()
print("done:", query.lastProgress["numInputRows"] if query.lastProgress else 0, "rows")

In [0]:
# CELL 2 - verify the schema and eyeball the rows
spark.table("lumora.bronze.tiktok_ads_raw").printSchema()

display(spark.table("lumora.bronze.tiktok_ads_raw").limit(5))

In [0]:
# CELL 3 - counts, coverage, and data-quality checks
spark.sql("""
  SELECT count(*) AS rows,
         count(DISTINCT _source_file) AS files,
         sum(CASE WHEN _rescued_data IS NOT NULL THEN 1 ELSE 0 END) AS rescued_rows
  FROM lumora.bronze.tiktok_ads_raw
""").display()

# Which files arrived, and how many rows each contributed?
spark.sql("""
  SELECT regexp_extract(_source_file, '([^/]+)$', 1) AS file_name,
         count(*) AS rows
  FROM lumora.bronze.tiktok_ads_raw
  GROUP BY 1 ORDER BY 1
""").display()

# The backfill overlap: same day + campaign appearing more than once
spark.sql("""
  SELECT data, id_da_campanha, count(*) AS n
  FROM lumora.bronze.tiktok_ads_raw
  GROUP BY 1, 2
  HAVING count(*) > 1
  ORDER BY 1
""").display()